# 1.0 Introduction
This notebook attempt to do the ETL and EDA of the FAO data on coffee for the top 15 producing countries over the period of 2000-2024.

## 1.1 Get all the necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid') # sets a white background with grid lines 
import plotly.express as px

In [2]:
# System and OS related tasks
import sys
import os
import importlib
# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

---

# 2.0 Data Source: FAO Data & ETL Process

## 2.1 FAO Source Data

FAO data is downloaded from [Food and Agriculture Organization of the United Nations](https://www.fao.org/faostat/en/#data/QCL)

It is deemed to be more expeditious to use the FAO Bulk Download option for their normalised data (1 row per country-year). The csv file has the following columns:


| Area Code | Area Code (M49) | Area | Item Code | Item Code (CPC) | Item | Element Code | Element | Year Code | Year | Unit | Value | Flag | Note |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|   |   |   |   |   |   |   |   |   |   |   |   |   |   |
|   |   |   |   |   |   |   |   |   |   |   |   |   |   |


We will subset the data based on 
* year (numeric) for 2000 to 2024 inclusive
* Item code (text field)
* Area (i.e. Country)


---

## 2.2 Understanding Key FAO Data Columns

#### 2.2.1 By Year

According to the FAO website, their data was updated in December 2025. As they sometimes provide an estimate of the data if they are not available, we will assume that the period of analysis to be from year 2000 to 2024 inclusive.

In the second half of the ETL, we might further restrict the number of years depending on the availablity of the World's BAnk Climate and Economics data.

---

#### 2.2.2 By Item Code

The items that this project is interested in are:
* "Crops, primary > (List) > Coffee, green" (**Item Code 656**)
  * There is only 1 item code of coffee listed.
  * Choosing the data granularity at the unprocessed level allows like for like comparison with the coffee data.
  * ‼️‼️📝 Note: Please see in the analysis of crop dominance the need to change the granularity of the coffee and tea data.

---

#### 2.2.3 By Area (Country)

We might only concentrate on 10-15 countries as part of the MVP for this project.
The country name will be converted to ISO3 country name convention using the handy `country_converter` package.

---

## 2.3 Extraction & Load FAO Data

In [3]:
# Read in the whole FAO Bulk Data file
raw_fao_dir = '../data/raw/Production_Crops_Livestock_E_All_Data_Normalized/'
filename = "Production_Crops_Livestock_E_All_Data_Normalized.csv"

data_columns = ['Area_Code', 'Area_Code_M49', 'Area', 'Item_Code', 'Item_Code_CPC', 'Item', 'Element_Code', 'Element', 'Year_Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']
column_dtypes = {
    'Area_Code': int, 'Area_Code_M49': str, 'Area' : str, 'Item_Code' : int, 'Item_Code_CPC': str, 'Item' : str, 'Element_Code' : int, 'Element' : str, 'Year_Code' : int, 'Year': int, 'Unit' : str, 'Value' : float, 'Flag' : str, 'Note' : str
}

df_fao_raw = pd.read_csv(
    f"{raw_fao_dir}/{filename}",
    header=0,
    names=data_columns,
    dtype=column_dtypes
    )
df_fao_raw.head(3)

,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Year_Code,Year,Unit,Value,Flag,Note
0,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1961,1961,ha,0.0,A,NaN
1,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1962,1962,ha,0.0,A,NaN
2,2,'004,Afghanistan,221,'01371,"Almonds, in shell",5312,Area harvested,1963,1963,ha,0.0,A,NaN


In [4]:
df_fao_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 4209110 entries, 0 to 4209109
Data columns (total 14 columns):
 #   Column         Dtype  
---  ------         -----  
 0   Area_Code      int64  
 1   Area_Code_M49  str    
 2   Area           str    
 3   Item_Code      int64  
 4   Item_Code_CPC  str    
 5   Item           str    
 6   Element_Code   int64  
 7   Element        str    
 8   Year_Code      int64  
 9   Year           int64  
 10  Unit           str    
 11  Value          float64
 12  Flag           str    
 13  Note           str    
dtypes: float64(1), int64(5), str(8)
memory usage: 449.6 MB


In [25]:
# Remove the first character (the apostrophe) in the Area_Code_M49 and the Item_Code_CPC
df_fao_raw['Area_Code_M49'] = df_fao_raw['Area_Code_M49'].str.strip("'")
df_fao_raw['Item_Code_CPC'] = df_fao_raw['Item_Code_CPC'].str.strip("'")
df_fao_raw.head(3)

,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Year_Code,Year,Unit,Value,Flag,Note
0,2,004,Afghanistan,221,01371,"Almonds, in shell",5312,Area harvested,1961,1961,ha,0,A,NaN
1,2,004,Afghanistan,221,01371,"Almonds, in shell",5312,Area harvested,1962,1962,ha,0,A,NaN
2,2,004,Afghanistan,221,01371,"Almonds, in shell",5312,Area harvested,1963,1963,ha,0,A,NaN


---

# 3.0 FAO Coffee Data

In [5]:
# Show all rows in value_counts output
pd.set_option('display.max_rows', None)

## 3.1 Coffee Items
All the entries with "coffee" in its various forms:

In [26]:
coffee_items = df_fao_raw['Item'].str.contains('coffee',  na=False, case=False)
df_coffee_items = df_fao_raw.loc[coffee_items]

print(df_coffee_items[['Item_Code','Item']].value_counts())

Item_Code  Item         
656        Coffee, green    20209
Name: count, dtype: int64


### 3.1.1 EDA of FAO Coffee Data

#### 3.1.1.1 EDA of FAO Coffee Data: Year and Item_Code Columns

Subset all the coffee data based on `Item_Code` = 656 and also for `Year` 2000-2024 inclusive.

In [27]:
df_fao_coffee = df_fao_raw[
    (df_fao_raw['Item_Code'] == 656) &
    ((df_fao_raw['Year'] >= 2000) &
    (df_fao_raw['Year'] <= 2024))
]

As the FAO bulk data file "Production_Crops_Livestock_E_All_Data_Normalized.csv" is 500+Mb in size, we will export the df_fao_coffee as the cvs raw data for audit purposes.

In [28]:
df_fao_coffee.to_csv(f'{raw_dir}/df_fao_coffee.csv', index=False)
print(f"Exported: {raw_dir}/df_fao_coffee.csv")

Exported: ../data/raw/df_fao_coffee.csv


🔍 Now taking a look at the `Year` Column:
* noted there might be extraeous rows in years in years 2005, 2000-2003 and fewer rows in rest of the years.
* this might even out after we look at the top 15 producting countries.

In [29]:
df_fao_coffee['Year'].value_counts()

Year
2000    320
2001    320
2002    320
2003    320
2005    320
2004    319
2006    319
2015    316
2016    316
2017    316
2011    315
2012    315
2013    315
2014    315
2008    314
2009    314
2010    314
2007    313
2023    313
2018    312
2019    312
2020    312
2021    312
2022    312
2024    310
Name: count, dtype: int64

🔍 Now taking a look at the `Item_Code` Column:

In [30]:
print("The List of Item Code in df_fao_coffee:")
print(df_fao_coffee['Item_Code'].value_counts())


The List of Item Code in df_fao_coffee:
Item_Code
656    7884
Name: count, dtype: int64


---

#### 3.1.1.2 EDA of FAO coffee Data: Area

🔍 Now taking a look at the `Area` & `Area_Code_M49' Columns:
* We can see that the FAO bulk data has aggregation at "World" and other region areas level. 
* As we are look at the country granularity, we will need to explicitly exclude these area aggregation granularity when selecting the top 15 producing countries.

In [40]:
print("The List of Countries or Regions in coffee:")
print(df_fao_coffee[['Area_Code_M49','Area']].value_counts())

The List of Countries or Regions in coffee:
Area_Code_M49  Area                                            
024            Angola                                              75
084            Belize                                              75
204            Benin                                               75
068            Bolivia (Plurinational State of)                    75
076            Brazil                                              75
108            Burundi                                             75
116            Cambodia                                            75
120            Cameroon                                            75
140            Central African Republic                            75
159            China                                               75
156            China, mainland                                     75
170            Colombia                                            75
174            Comoros                              

---

#### 3.1.1.3 EDA of FAO Coffee Data: Element

🔍 Now taking a look at the `Element`:
* The element values of are the measures for the respective country's output for the year. 
  * "Area harvested" (in hectares), 
  * "Production" (in tonnages) and 
  * "Yield" (in kilograms/hectare) etc 
* It is to be read in conjection with the `Unit` (unit of measurement) and the `Value` columns.
* In the Transformation section of for this Coffee data, we will do a check on the "Yield" column using the provided "Area harvested" and "Production".

In [15]:
print("The List of Elements in df_fao_coffee:")
print(df_fao_coffee[['Element', 'Unit']].value_counts())

The List of Elements in df_fao_coffee:
Element         Unit 
Production      t        2710
Area harvested  ha       2638
Yield           kg/ha    2536
Name: count, dtype: int64


---

## 3.2 Transformation of FAO Coffee Data and Export to CSV

### 3.2.1 Transformation of FAO coffee Data

The FAO bulk data file also include aggregrations rows in the data. They have been identified and as they are of different granularity to the country level data required by this project, we will need to remove them.

In [35]:
# create an exclude list for regions in the Area column 
exclude_areas = ['001'    # World
                , '002'    # Africa
                , '014'    # Eastern Africa
                , '017'    # Middle Africa
                , '018'    # Southern Africa
                , '011'    # Western Africa
                , '019'    # Americas
                , '013'    # Central America
                , '005'    # South America
                , '142'    # Asia
                , '030'    # Eastern Asia
                , '034'    # Southern Asia
                , '035'    # South-eastern Asia
                , '145'    # Western Asia
                , '150'    # Europe
                , '151'    # Eastern Europe
                , '039'    # Southern Europe
                , '009'    # Oceania
                , '054'    # Melanesia
                , '199'    # Least Developed Countries (LDCs)
                , '432'    # Land Locked Developing Countries (LLDCs)
                , '722'    # Small Island Developing States (SIDS)
                , '901'    # Low Income Food Deficit Countries (LIFDCs)
                , '902'    # Net Food Importing Developing Countries (NFIDCs)
                , '097'    # European Union (27)
                , '638'    # Réunion
                , '159'    # China (This code represents the aggregate of all Chinese territories for statistical purposes. It is the sum of Mainland China (156), Taiwan Province (214), Hong Kong SAR (96), and Macao SAR (128).)
]

exclude_areas

['001',
 '002',
 '014',
 '017',
 '018',
 '011',
 '019',
 '013',
 '005',
 '142',
 '030',
 '034',
 '035',
 '145',
 '150',
 '151',
 '039',
 '009',
 '054',
 '199',
 '432',
 '722',
 '901',
 '902',
 '097',
 '638',
 '159']

In [41]:
# new dataset to only contain countries (by using the ~)
df_fao_coffee_all_countries = df_fao_coffee[~df_fao_coffee['Area_Code_M49'].isin(exclude_areas)]

print("The List of Countries (exclude regions) in df_fao_coffee_all_countries:")
print(df_fao_coffee_all_countries['Area'].value_counts())

The List of Countries (exclude regions) in df_fao_coffee_all_countries:
Area
Angola                                75
Belize                                75
Benin                                 75
Bolivia (Plurinational State of)      75
Brazil                                75
Burundi                               75
Cambodia                              75
Cameroon                              75
Central African Republic              75
China, mainland                       75
Colombia                              75
Comoros                               75
Congo                                 75
Costa Rica                            75
Côte d'Ivoire                         75
Cuba                                  75
Democratic Republic of the Congo      75
Dominica                              75
Dominican Republic                    75
Ecuador                               75
El Salvador                           75
Equatorial Guinea                     75
Ethiopia             

As the MVP of this project is top 15 coffee countries by `Production` value, we need to sum the countries' production over the 25 year period in the data set.

In [42]:
# Sum up production by country
df_coffee_production_by_country = df_fao_coffee_all_countries[df_fao_coffee_all_countries['Element'] == 'Production'].groupby('Area')['Value'].sum().reset_index() 

# Rename the columns 
df_coffee_production_by_country.columns = ["Country", "Sum_Production_25yrs"]

# Take a look at the data and set display for the Sum_Production_25yrs
pd.set_option('display.float_format', '{:,.0f}'.format)
df_coffee_production_by_country.head(5)

,Country,Sum_Production_25yrs
0,Angola,"218,969"
1,Belize,"2,812"
2,Benin,"1,746"
3,Bolivia (Plurinational State of),"618,646"
4,Brazil,"68,927,744"


In [37]:
# sort the countries from high to low for column Sum_Production_25yrs
df_top_15_coffee_country_only = df_coffee_production_by_country.sort_values(
        "Sum_Production_25yrs", ascending=False
        ).head(15).reset_index(drop=True)

df_top_15_coffee_country_only

,Country,Sum_Production_25yrs
0,Brazil,"68,927,744"
1,Viet Nam,"32,900,417"
2,Colombia,"17,288,093"
3,Indonesia,"17,225,004"
4,Ethiopia,"9,713,167"
5,India,"7,659,875"
6,Honduras,"7,428,610"
7,Peru,"7,035,345"
8,Uganda,"6,256,771"
9,Guatemala,"6,116,237"


In [43]:
# Get top 15 countries data only
df_fao_coffee_top15_countries = df_fao_coffee_all_countries[
        df_fao_coffee_all_countries["Area"]
        .isin(df_top_15_coffee_country_only["Country"].unique())]

df_fao_coffee_top15_countries.head()


,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Year_Code,Year,Unit,Value,Flag,Note
337741,21,076,Brazil,656,01610,"Coffee, green",5312,Area harvested,2000,2000,ha,"2,267,968",A,NaN
337742,21,076,Brazil,656,01610,"Coffee, green",5312,Area harvested,2001,2001,ha,"2,336,031",A,NaN
337743,21,076,Brazil,656,01610,"Coffee, green",5312,Area harvested,2002,2002,ha,"2,370,891",A,NaN
337744,21,076,Brazil,656,01610,"Coffee, green",5312,Area harvested,2003,2003,ha,"2,395,501",A,NaN
337745,21,076,Brazil,656,01610,"Coffee, green",5312,Area harvested,2004,2004,ha,"2,368,040",A,NaN


In [44]:
# Double check the extraction only has 15 coffee countries
df_fao_coffee_top15_countries["Area"].value_counts()

Area
Brazil           75
Colombia         75
Costa Rica       75
Côte d'Ivoire    75
Ethiopia         75
Guatemala        75
Guinea           75
Honduras         75
India            75
Indonesia        75
Mexico           75
Peru             75
Uganda           75
Viet Nam         75
Nicaragua        72
Name: count, dtype: int64

‼️ Note that Nicaragua only has 72 rows of data instead of the expected 75. This should be investigated.

In [45]:
# Check that each of the top 15 Coffee countries has 3 of the `Element` measures
elements_by_country = df_fao_coffee_top15_countries.groupby("Area")["Element"].unique().reset_index()

print("Elements available for each of the top 15 countries")
print("===================================================")
for index, row in elements_by_country.iterrows():
    print(f"For The Country of {row['Area']}:")
    for element in row["Element"]:
        print(f" * {element}")
    print("-------------------------")    

Elements available for each of the top 15 countries
For The Country of Brazil:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Colombia:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Costa Rica:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Côte d'Ivoire:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Ethiopia:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Guatemala:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Guinea:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Honduras:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of India:
 * Area harvested
 * Yield
 * Production
-------------------------
For The Country of Indonesia:
 * Area harvested
 * Yield
 * Production
------------

---

### 3.2.2 Convert the Country Column into ISO3 Standard

In [46]:
import country_converter as coco

cc = coco.CountryConverter()

df_fao_coffee_top15_countries["Country_ISO3"] = cc.pandas_convert(
    series=df_fao_coffee_top15_countries['Area'],
    to='ISO3'
)

df_fao_coffee_top15_countries["Country_ISO3"].value_counts()

Country_ISO3
BRA    75
COL    75
CRI    75
CIV    75
ETH    75
GTM    75
GIN    75
HND    75
IND    75
IDN    75
MEX    75
PER    75
UGA    75
VNM    75
NIC    72
Name: count, dtype: int64

---

### 3.2.3 Export FAO Coffee Data to CSV for Analysis

In [47]:
df_fao_coffee_top15_countries.to_csv(f'{processed_dir}/df_fao_coffee_top15_countries.csv', index=False)
print(f"Exported: {processed_dir}/df_fao_coffee_top15_countries.csv")

Exported: ../data/processed/df_fao_coffee_top15_countries.csv


---